# Notebook 02 — Text Processing

**Goal:** Prepare the canonical Qaloon text for use as CTC training labels
and build the character-level vocabulary.

**Why this notebook exists:**
Raw Arabic text from `qalon_canonical.json` contains punctuation, tatweel,
and Unicode edge cases that must be removed before we can tokenize at the
character level.  We also need to decide *which* characters become vocabulary
tokens — this is the output vocabulary of the CTC model.  Getting this right
is critical: if a diacritic is missing from the vocab, the model can never
predict it.

**Output:**
- `data/processed/labels.json` — segment ID → normalized text
- `data/processed/vocab.json`  — character → index mapping

## Step 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/iqraa-ai')
!pip install -q -r /content/drive/MyDrive/iqraa-ai/requirements.txt

## Step 1 — Configure paths

In [ ]:
from pathlib import Path

REPO_ROOT     = Path('/content/drive/MyDrive/iqraa-ai')
TEXT_FILE     = REPO_ROOT / 'data' / 'text' / 'qalon_canonical.json'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
LABELS_OUT    = PROCESSED_DIR / 'labels.json'
VOCAB_OUT     = PROCESSED_DIR / 'vocab.json'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Step 2 — Load canonical text

We load all 6 214 ayahs from `qalon_canonical.json`.  Each entry has `surah`,
`ayah`, and `text` fields.

In [ ]:
import json

with open(TEXT_FILE, encoding='utf-8') as f:
    ayahs = json.load(f)

print(f'Loaded {len(ayahs)} ayahs')
print('Example:', ayahs[0])

## Step 3 — Apply Qaloon-specific normalization

We call `src.normalize_text.normalize_corpus()` which removes punctuation and
tatweel while preserving all diacritics.  The original text is kept under the
`text` key; the cleaned version is added as `text_normalized`.

In [ ]:
from src.normalize_text import normalize_corpus

ayahs = normalize_corpus(ayahs, keep_tashkeel=True)

print('Before:', ayahs[0]['text'])
print('After :', ayahs[0]['text_normalized'])

## Step 4 — Build segment ID → text label mapping

We create a dict keyed by `<surah_padded>_<ayah_padded>` matching the WAV
filename convention from Notebook 01.  This is the file that `src/dataset.py`
reads to pair audio with labels.

In [ ]:
labels = {
    f"{a['surah']:03d}_{a['ayah']:03d}": a['text_normalized']
    for a in ayahs
}

with open(LABELS_OUT, 'w', encoding='utf-8') as f:
    json.dump(labels, f, ensure_ascii=False, indent=2)

print(f'Saved {len(labels)} labels to {LABELS_OUT}')

## Step 5 — Build the character vocabulary

We call `src.build_vocab.build_vocab()` which scans all normalized texts,
collects unique characters, and prepends the three special tokens required
by Wav2Vec2ForCTC: `[PAD]` (blank), `[UNK]`, and `|` (word boundary).

In [ ]:
from src.build_vocab import build_vocab

texts = [a['text_normalized'] for a in ayahs]
vocab = build_vocab(texts, output_path=str(VOCAB_OUT))

print(f'Vocabulary size: {len(vocab)} tokens')
print('First 10 tokens:', list(vocab.items())[:10])

## Step 6 — Inspect diacritic coverage

Verify that all expected Arabic diacritics are present in the vocabulary.
If any are missing the model cannot predict them.

In [ ]:
TASHKEEL = list('\u064b\u064c\u064d\u064e\u064f\u0650\u0651\u0652\u0670')
NAMES    = ['tanwin_fath','tanwin_damm','tanwin_kasr','fatha','damma',
            'kasra','shadda','sukun','superscript_alif']

for diac, name in zip(TASHKEEL, NAMES):
    status = '✓' if diac in vocab else '✗ MISSING'
    print(f'  {status}  U+{ord(diac):04X}  {name}')